# 03 — Silver: Green Taxi

Reads Green Taxi records from Bronze (`tlc_bronze.green_raw`), applies data
quality rules via **flag-based annotation** (no silent drops), enriches with
zone metadata, builds the normalized Silver schema, and writes:
- **Valid records** → `tlc_silver.trips_green` (with `quality_flags` preserved)
- **Invalid records** → `tlc_audit.quarantine` (with `_rejection_reason`)

Silver `_meta` carries `bronze_run_id` + `source_file` for full lineage tracing.

> **Rules note:** Green uses `GREEN_RULES` = `YELLOW_GREEN_SHARED_RULES` (no datetime columns)
> + `GREEN_EXTRA_RULES` (`lpep_*` time order).  
> **Never** apply `YELLOW_RULES` or `YELLOW_GREEN_RULES` to Green data — they reference
> `tpep_*` columns that don't exist in Green DataFrames.

## Imports & Configuration

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work')

from src.spark_utils import get_spark, read_mongo, write_mongo
from src.transformations.tlc_rules import (
    GREEN_RULES,                   # = YELLOW_GREEN_SHARED_RULES + GREEN_EXTRA_RULES
    apply_quality_flags,
    route_quarantine,
    print_rejection_summary,
)
from src.transformations.enrichment import load_zone_lookup, enrich_trip_locations
from src.transformations.schema import build_green_silver
from core.config.settings import settings
from core.audit.control_manager import ControlManager, ExecutionStatus
from core.storage.mongo_client import get_audit_db
import pyspark.sql.functions as F
import datetime

YEARS_TO_PROCESS = [2024, 2025]
print("Imports OK.")

## Start Spark & Audit Control

In [ ]:
spark = get_spark(app_name="TLC_Silver_Green")

control = ControlManager("silver_green")
execution = control.start({"years": YEARS_TO_PROCESS, "vehicle_type": "green"})
run_id = execution.execution_id
print(f"Execution ID (Silver run_id): {run_id}")

## Read from Bronze

In [ ]:
df_bronze = read_mongo(spark, settings.MONGO_DB_BRONZE, "green_raw")

if "_meta" in df_bronze.columns:
    df_bronze = df_bronze.filter(F.year("lpep_pickup_datetime").isin(YEARS_TO_PROCESS))

# records_in will be evaluated after caching df_flagged


## Apply Data Quality Flags

`GREEN_RULES` contains:
- `valid_distance`, `valid_fare`, `valid_total` — numeric sanity (no datetime columns)
- `valid_pickup_zone`, `valid_dropoff_zone` — location ID in [1, 265]
- `valid_passengers` — passenger count in [1, 8]
- `valid_time_order_green` — `lpep_dropoff > lpep_pickup` (Green-specific)

In [ ]:
df_flagged = apply_quality_flags(df_bronze, GREEN_RULES)
# OPTIMIZATION: Cache df_flagged to avoid double-evaluating Bronze DAG
df_flagged.cache()
records_in = df_flagged.count()
print(f"Records read from Bronze: {records_in:,}")

valid_df, rejected_df = route_quarantine(df_flagged)

records_valid    = valid_df.count()
records_rejected = rejected_df.count()

print(f"Valid records   : {records_valid:,}")
print(f"Rejected records: {records_rejected:,}")
print(f"Quarantine rate : {records_rejected / records_in * 100:.2f}%" if records_in else "N/A")

print_rejection_summary(rejected_df)

control.log_quality_check(
    check_name="green_quality_flags",
    dataset=f"green_raw_years_{YEARS_TO_PROCESS}",
    records_checked=records_in,
    records_failed=records_rejected,
)

## Route Rejected Records to Quarantine

In [ ]:
if records_rejected > 0:
    # OPTIMIZATION: Distributed PySpark write instead of driver collect()
    raw_cols = [c for c in rejected_df.columns if c not in ("_rejection_reason", "quality_flags", "_meta")]
    
    quarantine_df_mapped = rejected_df.select(
        F.current_timestamp().alias("quarantined_at"),
        F.lit(run_id).alias("silver_run_id"),
        F.col("_meta.run_id").alias("bronze_run_id"),
        F.col("_meta.source_file").alias("source_file"),
        F.lit("silver_green").alias("pipeline"),
        F.col("_rejection_reason").alias("reason"),
        F.col("quality_flags").alias("quality_flags"),
        F.struct(*[F.col(c) for c in raw_cols]).alias("raw_record")
    )
    
    write_mongo(quarantine_df_mapped, settings.MONGO_DB_AUDIT, "quarantine", mode="append")
    print(f"Quarantined {records_rejected:,} records into tlc_audit.quarantine (Distributed)")
else:
    print("No records quarantined.")


## Enrich with Zone Metadata

In [ ]:
zone_df = load_zone_lookup(spark)
valid_df = enrich_trip_locations(valid_df, zone_df)
print("Zone enrichment complete.")

## Build Silver Schema & Write to MongoDB

In [ ]:
silver_df = build_green_silver(valid_df, run_id=run_id)
n_written = write_mongo(silver_df, settings.MONGO_DB_SILVER, "trips_green", mode="append", num_rows=records_valid)
print(f"Rows written to tlc_silver.trips_green: {n_written:,}")

## Volumetric Traceability Check

In [ ]:
print(f"╔══════════════════════════════════════════╗")
print(f"║  Volumetric Traceability Report          ║")
print(f"╠══════════════════════════════════════════╣")
print(f"║  Bronze records in  : {records_in:>16,}  ║")
print(f"║  Silver records out : {n_written:>16,}  ║")
print(f"║  Quarantine records : {records_rejected:>16,}  ║")
print(f"║  Delta (must be 0)  : {records_in - n_written - records_rejected:>16,}  ║")
print(f"╚══════════════════════════════════════════╝")
assert records_in == n_written + records_rejected, "DATA LOSS DETECTED — investigate immediately!"

df_flagged.unpersist()
print("Cache cleared.")

control.end(
    ExecutionStatus.SUCCESS,
    {
        "records_read_from_bronze": records_in,
        "records_written_to_silver": n_written,
        "records_quarantined": records_rejected,
        "quarantine_rate_pct": round(records_rejected / records_in * 100, 4) if records_in else 0,
    },
)

## Audit Report

In [ ]:
import json
print(json.dumps(control.get_report(), indent=2, default=str))